# Solar Filament Segmentation Challenge 2026 — Baseline Model

This notebook adapts the solution notebook's semantic segmentation workflow for the complete MAGFiLO training set. It uses a pretrained ResNet-34 U-Net, binary cross-entropy plus Dice loss, image-level train/validation splitting, early stopping, validation threshold selection, and connected-component RLE submission generation. Source images are resized to 512×512 for training and predictions are resized back to each image's original dimensions before evaluation or submission.

Repeated annotation records for the same JPEG are retained as separate training examples, but all records for a filename are assigned to the same split. This prevents an image from appearing in both training and validation. The competition submission contains one row per predicted connected filament component.


## 1. Install dependencies and import libraries

Run the install cell once in a fresh Kaggle session. The pretrained encoder weights may be downloaded the first time the model is created.

In [ ]:
%pip install -q segmentation-models-pytorch albumentations pycocotools

In [ ]:
from pathlib import Path
import os
import random
import warnings

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configure paths and experiment

The dataset finder supports the local project folder and Kaggle's mounted competition datasets. Adjust the training settings here to fit the available accelerator memory.

In [ ]:
SEED = 42
IMAGE_SIZE = 512
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
VALIDATION_FRACTION = 0.20
PATIENCE = 6
MIN_COMPONENT_AREA = 20
OUTPUT_DIR = Path.cwd()
CHECKPOINT_PATH = OUTPUT_DIR / "best_filament_unet.pth"
SUBMISSION_PATH = OUTPUT_DIR / "solar_filament_submission.csv"
CREATE_SUBMISSION = True
NUM_WORKERS = 0 if os.name == "nt" else 2

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    for parent in candidates:
        if not parent.exists():
            continue
        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train" / "train_images").is_dir() and (root / "test" / "test_images").is_dir():
                return root
    raise FileNotFoundError("Could not find MAGFiLO_1.0_Kaggle_2026 with train and test image folders.")

DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")

## 3. Load annotations and create a leakage-safe split

The COCO records refer to 707 physical JPEGs, with some JPEGs appearing in multiple annotator records. We split by filename, then retain each record as an individual sample within its assigned split.

In [ ]:
coco = COCO(str(ANNOTATION_PATH))
image_ids = list(coco.imgs.keys())
records = pd.DataFrame([
    {"image_id": image_id, "file_name": coco.imgs[image_id]["file_name"]}
    for image_id in image_ids
])
splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(records, groups=records["file_name"]))
train_ids = records.iloc[train_idx]["image_id"].tolist()
val_ids = records.iloc[val_idx]["image_id"].tolist()
train_files = set(records.iloc[train_idx]["file_name"])
val_files = set(records.iloc[val_idx]["file_name"])
assert train_files.isdisjoint(val_files), "A physical image appears in both splits."
print(f"Training records: {len(train_ids)} across {len(train_files)} images")
print(f"Validation records: {len(val_ids)} across {len(val_files)} images")
print(f"Train/validation filename overlap: {len(train_files & val_files)}")

## 4. Prepare augmentation and dataset loaders

Images are grayscale. Training transforms apply shared geometric operations to the image and binary union mask, then apply mild intensity variation to the image only. Validation uses resizing and normalization only.

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.08, rotate_limit=12,
                       border_mode=cv2.BORDER_CONSTANT, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

class FilamentDataset(Dataset):
    def __init__(self, coco_api, ids, image_dir, transform):
        self.coco = coco_api
        self.ids = ids
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        info = self.coco.loadImgs([image_id])[0]
        image_path = self.image_dir / info["file_name"]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise OSError(f"Could not read image: {image_path}")
        mask = np.zeros(image.shape, dtype=np.uint8)
        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        for annotation in self.coco.loadAnns(ann_ids):
            mask |= self.coco.annToMask(annotation).astype(np.uint8)
        augmented = self.transform(image=image[..., None], mask=mask)
        image_tensor = augmented["image"].float()
        mask_tensor = augmented["mask"].unsqueeze(0).float()
        return image_tensor, mask_tensor

train_dataset = FilamentDataset(coco, train_ids, TRAIN_IMAGE_DIR, train_transform)
val_dataset = FilamentDataset(coco, val_ids, TRAIN_IMAGE_DIR, val_transform)
loader_args = {"batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
               "pin_memory": DEVICE.type == "cuda"}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)
print(f"Batches: train={len(train_loader)}, validation={len(val_loader)}")

## 5. Build the pretrained U-Net and loss

A one-channel ResNet-34 U-Net predicts foreground logits. BCEWithLogitsLoss handles pixel-wise classification, while soft Dice loss emphasizes overlap for the sparse filament pixels.

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None,
).to(DEVICE)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probabilities = torch.sigmoid(logits).flatten(1)
        targets = targets.flatten(1)
        intersection = (probabilities * targets).sum(dim=1)
        dice = (2 * intersection + self.smooth) / (
            probabilities.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )
        return 1 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    predictions = (torch.sigmoid(logits) >= threshold).float()
    intersection = (predictions * targets).sum()
    return (2 * intersection + smooth) / (predictions.sum() + targets.sum() + smooth)

criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Train, validate, and save the best checkpoint

Each epoch runs a complete training pass and a no-gradient validation pass. Early stopping tracks validation Dice; the best checkpoint is saved whenever validation Dice improves. Mixed precision is enabled on CUDA to reduce memory use.

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_dice": [], "learning_rate": []}
best_val_dice = -1.0
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_total = 0.0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss_total += loss.item() * images.size(0)
    train_loss = train_loss_total / len(train_dataset)

    model.eval()
    val_loss_total, val_dice_total = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
            val_loss_total += loss.item() * images.size(0)
            val_dice_total += dice_score(logits, masks).item() * images.size(0)
    val_loss = val_loss_total / len(val_dataset)
    val_dice = val_dice_total / len(val_dataset)
    scheduler.step()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["learning_rate"].append(optimizer.param_groups[0]["lr"])
    print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | train loss {train_loss:.4f} | "
          f"val loss {val_loss:.4f} | val Dice {val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save({"model_state_dict": model.state_dict(), "image_size": IMAGE_SIZE,
                    "best_val_dice": best_val_dice, "best_val_loss": best_val_loss}, CHECKPOINT_PATH)
        print(f"  Saved best checkpoint: {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {epoch + 1} epochs.")
            break

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Best validation Dice: {checkpoint['best_val_dice']:.4f}")

## 7. Select a foreground threshold on validation

The validation predictions are resized to each source image's original resolution before threshold scoring. Threshold candidates are compared using global pixel Dice. A few examples are retained for the final visual review.

In [ ]:
THRESHOLD_CANDIDATES = (0.25, 0.35, 0.45, 0.50, 0.60)
threshold_stats = {threshold: [0.0, 0.0] for threshold in THRESHOLD_CANDIDATES}
validation_examples = []

def prepare_image(image):
    resized = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    normalized = (resized.astype(np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(normalized).unsqueeze(0).unsqueeze(0)

def predict_probability(image):
    tensor = prepare_image(image).to(DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
        probability = torch.sigmoid(model(tensor))[0, 0].float().cpu().numpy()
    return cv2.resize(probability, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)

for index, image_id in enumerate(val_ids):
    info = coco.loadImgs([image_id])[0]
    image = cv2.imread(str(TRAIN_IMAGE_DIR / info["file_name"]), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"Could not read validation image: {info['file_name']}")
    target = np.zeros(image.shape, dtype=np.uint8)
    for annotation in coco.loadAnns(coco.getAnnIds(imgIds=[image_id])):
        target |= coco.annToMask(annotation).astype(np.uint8)
    probability = predict_probability(image)
    for threshold, stats in threshold_stats.items():
        prediction = probability >= threshold
        stats[0] += np.logical_and(prediction, target).sum()
        stats[1] += prediction.sum() + target.sum()
    if len(validation_examples) < 4:
        validation_examples.append({"image": image, "target": target, "probability": probability,
                                    "file_name": info["file_name"]})
    if (index + 1) % 25 == 0 or index + 1 == len(val_ids):
        print(f"Validated {index + 1}/{len(val_ids)} annotation records")

threshold_scores = {
    threshold: (2 * intersection + 1e-6) / (denominator + 1e-6)
    for threshold, (intersection, denominator) in threshold_stats.items()
}
BEST_THRESHOLD = max(threshold_scores, key=threshold_scores.get)
threshold_summary = pd.DataFrame([
    {"threshold": t, "pixel_dice": score} for t, score in threshold_scores.items()
]).sort_values("pixel_dice", ascending=False)
print(threshold_summary.to_string(index=False))
print(f"Selected threshold: {BEST_THRESHOLD:.2f}")

## 8. Generate the competition submission

Test images are predicted at 512×512 and restored to native size. Connected components become individual filament rows. The output follows the required `filament_id,segmentation_rle` schema and encodes masks in COCO's column-major RLE format.

In [ ]:
def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")

def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask.astype(np.uint8), connectivity=8)
    components = [labels == label for label in range(1, count)
                  if stats[label, cv2.CC_STAT_AREA] >= min_area]
    components.sort(key=lambda component: (-int(component.sum()), int(np.where(component)[0].min()),
                                           int(np.where(component)[1].min())))
    return components

def get_test_image_paths():
    paths = sorted(path for path in TEST_IMAGE_DIR.iterdir()
                   if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"})
    if not paths:
        raise FileNotFoundError(f"No test JPEGs found in {TEST_IMAGE_DIR}")
    if len({path.stem for path in paths}) != len(paths):
        raise ValueError("Test images contain duplicate filename stems.")
    return paths

def create_submission(output_path=SUBMISSION_PATH):
    rows = []
    paths = get_test_image_paths()
    for image_index, path in enumerate(paths, start=1):
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise OSError(f"Could not read test image: {path}")
        probability = predict_probability(image)
        binary_mask = probability >= BEST_THRESHOLD
        components = split_into_filaments(binary_mask)
        for component_index, component in enumerate(components, start=1):
            rows.append({"filament_id": f"{path.stem}_{component_index}",
                         "segmentation_rle": mask_to_rle(component)})
        if image_index % 25 == 0 or image_index == len(paths):
            print(f"Predicted {image_index}/{len(paths)} test images")
    submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    if submission["filament_id"].duplicated().any():
        raise ValueError("Generated submission contains duplicate filament IDs.")
    submission.to_csv(output_path, index=False)
    print(f"Saved {len(submission)} filament rows to {output_path}")
    return submission

if CREATE_SUBMISSION:
    submission_df = create_submission()
    display(submission_df.head())

## 9. Training graphs and predicted validation examples

The final section plots the loss and Dice histories, compares validation threshold candidates, and shows original images beside their target masks and predicted probability/binary masks. These cells run after training and submission creation.

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
axes[0].set(title="BCE + Dice loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(epochs, history["val_dice"], marker="o", color="tab:green")
axes[1].set(title="Validation Dice at 0.50", xlabel="Epoch", ylabel="Dice", ylim=(0, 1))
axes[2].plot(threshold_summary["threshold"], threshold_summary["pixel_dice"], marker="o", color="tab:purple")
axes[2].axvline(BEST_THRESHOLD, color="tab:red", linestyle="--", label=f"Selected: {BEST_THRESHOLD:.2f}")
axes[2].set(title="Validation threshold selection", xlabel="Foreground threshold", ylabel="Pixel Dice", ylim=(0, 1))
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
for example in validation_examples:
    image = example["image"]
    target = example["target"]
    probability = example["probability"]
    prediction = probability >= BEST_THRESHOLD
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Input image")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")
    axes[2].imshow(probability, cmap="magma", vmin=0, vmax=1)
    axes[2].set_title("Predicted probability")
    axes[3].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"Prediction (threshold {BEST_THRESHOLD:.2f})")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(example["file_name"])
    fig.tight_layout()
    plt.show()